# BME 544 — PPG Signal Quality & HRV Analysis

**Course:** BME 544 – Biomedical Signal Processing  
**Dataset:** `ppg_dataset/connors_heart.txt` (100 Hz, raw ADC values)

This notebook replays a recorded PPG signal, applies a custom signal-quality index (SQI) pipeline to reject noisy windows, detects systolic peaks using a bandpass + local-maxima algorithm, and computes short-term HRV (RMSSD) every 10 seconds over a rolling 30-second window.

**Pipeline overview:**
1. Custom PPG peak detection (no external HRV library)
2. Segment-level SQI gating (variance, range, flatline, path-length, peak plausibility)
3. RMSSD computed from cleaned RR intervals
4. Full session summary + signal visualization

In [ ]:
import os
import sys
import threading
import queue as queue_module
import time
import warnings
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Configuration  ← change only this block
# ---------------------------------------------------------------------------
DATA_FILE     = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                             "ppg_dataset", "connors_heart.txt")
SAMPLING_RATE = 100          # Hz
WINDOW_SEC    = 30           # sliding window length
UPDATE_SEC    = 10           # HRV recalculation cadence

print(f"Data file  : {DATA_FILE}")
print(f"Sample rate: {SAMPLING_RATE} Hz")
print(f"Window     : {WINDOW_SEC} s,  updated every {UPDATE_SEC} s")

## 1. Peak Detection & Signal Processing Utilities

A three-stage pipeline replaces the NeuroKit2 dependency:

1. **Bandpass filter** (2nd-order Butterworth, 0.5 – 4 Hz) removes baseline wander and high-frequency noise while keeping the cardiac band (30 – 240 bpm).  
2. **`find_peaks`** with a minimum inter-peak distance of 0.4 s (≤ 150 bpm) and an amplitude threshold of μ + 0.3σ on the filtered signal.  
3. **`clean_peaks`** removes spurious detections on rising edges and merges peaks that are unrealistically close, keeping the higher-amplitude one.

In [ ]:
def detect_ppg_peaks(signal, sampling_rate=100):
    """
    Detect systolic peaks in a raw PPG signal.

    Steps
    -----
    1. 2nd-order Butterworth bandpass (0.5 – 4 Hz) to isolate the cardiac band.
    2. scipy find_peaks with amplitude threshold (mean + 0.3*std of filtered
       signal) and a minimum inter-peak distance of 0.4 s (≤ 150 bpm).

    Returns
    -------
    peaks : np.ndarray
        1-D array of peak indices into *signal* (not the filtered copy).
    """
    sig = np.asarray(signal, dtype=float)
    if len(sig) < 10:
        return np.array([], dtype=int)

    nyq = sampling_rate / 2.0
    low  = max(0.5 / nyq, 1e-4)
    high = min(4.0 / nyq, 0.99)
    b, a = butter(2, [low, high], btype="band")
    filtered = filtfilt(b, a, sig)

    min_dist      = max(1, int(0.4 * sampling_rate))   # 40 samples @ 100 Hz
    height_thresh = np.mean(filtered) + 0.3 * np.std(filtered)

    peaks, _ = find_peaks(filtered, distance=min_dist, height=height_thresh)
    return peaks


def initial_cleaning(ppg_window, peak_indices, rising_window=5):
    """
    Remove peaks that sit on a rising edge, i.e. the next `rising_window`
    samples are strictly increasing (the true peak has not yet been reached).
    """
    cleaned = []
    for idx in peak_indices:
        end_idx = min(idx + rising_window + 1, len(ppg_window))
        after   = ppg_window[idx:end_idx]
        if len(after) > 1 and all(after[i] < after[i + 1] for i in range(len(after) - 1)):
            continue  # on a rising edge — skip
        cleaned.append(idx)
    return cleaned


def clean_peaks(ppg_window, peak_indices, rising_window=5, min_distance=20):
    """
    Two-pass peak cleaner:
      1. Drop peaks on rising edges (initial_cleaning).
      2. Merge peaks closer than `min_distance` samples, keeping the taller one.

    Parameters
    ----------
    ppg_window   : array-like  Raw PPG signal.
    peak_indices : array-like  Candidate peak indices (relative to ppg_window).
    rising_window: int         Samples to look ahead when checking rising edges.
    min_distance : int         Minimum allowed samples between any two peaks.

    Returns
    -------
    list of cleaned peak indices
    """
    cleaned = initial_cleaning(ppg_window, list(peak_indices), rising_window)

    i = 0
    while i < len(cleaned) - 1:
        if cleaned[i + 1] - cleaned[i] < min_distance:
            if ppg_window[cleaned[i]] >= ppg_window[cleaned[i + 1]]:
                cleaned.pop(i + 1)
            else:
                cleaned.pop(i)
        else:
            i += 1
    return cleaned


print("Peak detection utilities defined.")

## 2. HRV (RMSSD) Calculation

RMSSD (Root Mean Square of Successive Differences) is computed directly from
the inter-beat intervals (IBIs):

$$\text{RMSSD} = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N-1}(\text{IBI}_{i+1} - \text{IBI}_i)^2}$$

where each IBI is the time between consecutive systolic peaks in milliseconds.

In [ ]:
# Global accumulators shared between the streaming thread and main loop.
# Initialised/reset inside collect_and_analyze_from_file.
all_ppg_data            = []
ppg_window_data_combined  = []
ppg_peaks_data_combined   = []
appended_samples          = 0


def compute_rmssd(peak_indices, sampling_rate=100):
    """
    Compute RMSSD from an array of peak sample indices.

    Parameters
    ----------
    peak_indices  : array-like  Sorted sample indices of detected peaks.
    sampling_rate : int         Samples per second.

    Returns
    -------
    float or None  RMSSD in milliseconds, or None if < 3 peaks supplied.
    """
    peaks = np.asarray(peak_indices, dtype=float)
    if len(peaks) < 3:
        return None
    ibi_ms = np.diff(peaks) / sampling_rate * 1000.0   # inter-beat intervals (ms)
    successive_diffs = np.diff(ibi_ms)
    return float(np.sqrt(np.mean(successive_diffs ** 2)))


def calculate_hrv_rmssd(ppg_window, sampling_rate=100):
    """
    Detect peaks in a 30-second window, clean them, then compute RMSSD.

    Returns
    -------
    (rmssd, global_peak_indices)
        rmssd              : float or None
        global_peak_indices: list — peak positions relative to the full session
    """
    global appended_samples, ppg_window_data_combined

    if len(ppg_window) < sampling_rate * 10:
        return None, []

    step_samples = sampling_rate * 10
    new_samples  = ppg_window if appended_samples == 0 else ppg_window[-step_samples:]
    ppg_window_data_combined.extend(new_samples)
    appended_samples += len(new_samples)

    try:
        raw_peaks     = detect_ppg_peaks(ppg_window, sampling_rate)
        cleaned_peaks = clean_peaks(ppg_window, raw_peaks)
        rmssd         = compute_rmssd(cleaned_peaks, sampling_rate)

        if rmssd is None:
            return None, []

        window_start_index = appended_samples - len(ppg_window)
        global_peaks = [i + window_start_index for i in cleaned_peaks]
        return rmssd, global_peaks

    except Exception as exc:
        print(f"\n[WARNING] HRV calculation failed: {exc}")
        return None, []


def check_hrv_status(current_rmssd, previous_rmssd):
    """Classify change in RMSSD: 0=improving, 1=slight drop, 2=significant drop."""
    if previous_rmssd is None or current_rmssd is None:
        return 1
    change = current_rmssd - previous_rmssd
    if change >= 0:
        return 0
    elif change > -5:
        return 1
    return 2


def display_feedback(status, current_rmssd, previous_rmssd, window_number):
    """Print a formatted HRV status block to the notebook output."""
    status_info = {
        0: {"symbol": "✓", "color": "GREEN",  "message": "EXCELLENT — HRV IMPROVING"},
        1: {"symbol": "~", "color": "YELLOW", "message": "GOOD — SLIGHT DECREASE"},
        2: {"symbol": "✗", "color": "RED",    "message": "REFOCUS — SIGNIFICANT DROP"},
    }
    info = status_info.get(status, status_info[1])
    change_str = (f"{current_rmssd - previous_rmssd:+.2f} ms"
                  if previous_rmssd is not None else "N/A (first reading)")

    print(f"\n{'='*60}")
    print(f"Window #{window_number}")
    print(f"RMSSD: {current_rmssd:.2f} ms  |  Change: {change_str}")
    print(f"Status: [{info['symbol']}] {info['color']} — {info['message']}")
    print(f"{'='*60}")


print("HRV functions defined.")

## 3. Signal Quality Index (SQI) — Noise Detection

Each 30-second window is split into 3-second segments. A segment is rejected if it fails **any** of these checks:

| Check | Rationale |
|---|---|
| Empty segment | Defensive guard |
| Std-dev < 1.0 ADC | Flatline / signal lost |
| Peak-to-peak < 80 ADC | Too small to contain a valid pulse |
| Any sample = 0 or ≥ 4000 | Sensor saturation / hard limit |
| Unique-value ratio < 2% | Stuck ADC output |
| Max abs diff > 2000 ADC | Impulsive spike/motion artefact |
| Path length > 12 000 | High-frequency noise (experimentally tuned) |
| Peak count outside [2, 6] | Implausible HR for a 3-second segment |

If any segment in the window is bad, the entire window is discarded.

In [ ]:
def is_segment_bad(segment, sampling_rate):
    """
    Apply SQI heuristics to a short (~3 s) PPG segment.

    Returns True (bad) if ANY check fails, False (clean) otherwise.
    """
    if len(segment) == 0:
        return True

    seg = np.asarray(segment, dtype=float)

    # 1. Flatline / low variance
    if np.std(seg) < 1.0:
        print("[SQI] FAIL — flatline / low variance")
        return True

    # 2. Low peak-to-peak amplitude
    if np.ptp(seg) < 80:
        print("[SQI] FAIL — peak-to-peak range too small")
        return True

    # 3. Sensor hard limits (0 – 4000 ADC)
    if np.any(seg == 0) or np.any(seg >= 4000):
        print("[SQI] FAIL — value at sensor limit")
        return True

    # 4. Clipping / stuck output
    if len(np.unique(seg)) / len(seg) < 0.02:
        print("[SQI] FAIL — clipping / too many identical values")
        return True

    # 5. Impulsive artefact (extreme sample-to-sample jump)
    if np.max(np.abs(np.diff(seg))) > 2000:
        print("[SQI] FAIL — extreme sample jump")
        return True

    # 6. High path length (broadband noise)
    path_length = np.sum(np.abs(np.diff(seg)))
    if path_length > 12_000:
        print(f"[SQI] FAIL — path length {path_length:.0f} > 12 000")
        return True

    # 7. Peak plausibility (custom detector, no neurokit)
    try:
        peaks = detect_ppg_peaks(seg, sampling_rate)
        if len(peaks) < 2 or len(peaks) > 6:
            print(f"[SQI] FAIL — {len(peaks)} peaks in 3 s segment (expected 2–6)")
            return True
    except Exception:
        return True

    return False


def is_window_bad(ppg_window, sampling_rate, segment_sec=3, max_bad_segments=0):
    """
    Split `ppg_window` into non-overlapping `segment_sec`-second chunks and
    reject the entire window if more than `max_bad_segments` are bad.

    With the default max_bad_segments=0, any single bad segment rejects the window.
    """
    seg_len = int(segment_sec * sampling_rate)
    if seg_len <= 0:
        return True

    bad_count = 0
    n_segments = len(ppg_window) // seg_len

    for i in range(n_segments):
        start   = i * seg_len
        segment = ppg_window[start : start + seg_len]
        if is_segment_bad(segment, sampling_rate):
            bad_count += 1
            print(f"[SQI] Bad segment #{i + 1} of {n_segments}")
            if bad_count > max_bad_segments:
                return True

    return False


print("SQI functions defined.")

## 4. Data Pipeline

The file is read in a background thread that pushes one sample at a time into a thread-safe queue, mirroring how live BLE data would arrive.  The main loop drains the queue, maintains a rolling buffer, and triggers HRV calculation every 10 seconds once the initial 30-second window has accumulated.

In [ ]:
def _file_stream_thread(file_path, data_queue, stop_event, connected_event,
                        sampling_rate=100, realtime=False):
    """
    Read one integer per line from file_path and enqueue each value.

    realtime=True  → sleeps 1/sampling_rate between samples (actual tempo).
    realtime=False → pushes as fast as possible (good for quick analysis).
    """
    try:
        with open(file_path, "r") as fh:
            lines = [ln.strip() for ln in fh if ln.strip()]
    except Exception as exc:
        print(f"[ERROR] Could not open file: {exc}")
        return

    connected_event.set()
    interval = 1.0 / sampling_rate if realtime else 0.0

    for line in lines:
        if stop_event.is_set():
            break
        try:
            data_queue.put(int(line))
        except ValueError:
            pass
        if interval:
            time.sleep(interval)

    stop_event.set()  # signal EOF to the main loop


def collect_and_analyze_from_file(file_path, duration, realtime=False,
                                  sampling_rate=SAMPLING_RATE,
                                  window_sec=WINDOW_SEC,
                                  update_sec=UPDATE_SEC):
    """
    Replay `file_path` and run the full HRV pipeline.

    Returns a dict with:
        all_ppg       – every raw sample
        peak_indices  – global peak positions (sample index)
        rmssd_log     – list of (time_s, rmssd) tuples
        status_counts – dict {0, 1, 2} → window count
    """
    global all_ppg_data, ppg_window_data_combined, ppg_peaks_data_combined, appended_samples

    window_samples = window_sec * sampling_rate

    data_queue     = queue_module.Queue()
    stop_event     = threading.Event()
    connected_event = threading.Event()

    file_thread = threading.Thread(
        target=_file_stream_thread,
        args=(file_path, data_queue, stop_event, connected_event),
        kwargs={"sampling_rate": sampling_rate, "realtime": realtime},
        daemon=True,
    )

    # Reset global state
    all_ppg_data             = []
    ppg_window_data_combined = []
    ppg_peaks_data_combined  = []
    appended_samples         = 0

    rmssd_log     = []          # (time_s, rmssd)
    status_counts = {0: 0, 1: 0, 2: 0}
    previous_rmssd        = None
    last_calculation_time = 0.0
    last_peak_sample      = -1
    window_count          = 0
    sample_count          = 0

    print(f"\n{'='*60}")
    print(f"Loading: {file_path}")
    file_thread.start()

    if not connected_event.wait(timeout=10):
        print("[ERROR] File could not be opened.")
        stop_event.set()
        return {}

    mode = "real-time (100 Hz)" if realtime else "fast (no sleep)"
    print(f"Replaying in {mode} mode — duration cap: {duration} s")
    print(f"HRV every {update_sec} s over {window_sec}-s windows")
    print(f"{'='*60}\n")

    while True:
        # Drain queue
        try:
            while True:
                val = data_queue.get_nowait()
                all_ppg_data.append(val)
                sample_count += 1
                if sample_count % 500 == 0:
                    print(f"  Samples: {sample_count} | Time: {sample_count/sampling_rate:.1f} s",
                          end="\r")
        except queue_module.Empty:
            pass

        time.sleep(0.001)

        data_time = sample_count / sampling_rate

        while (data_time >= window_sec
               and data_time - last_calculation_time >= update_sec):

            next_calc = last_calculation_time + update_sec
            last_calculation_time = next_calc

            if next_calc < window_sec:
                continue

            window_count += 1
            w_end   = int(next_calc * sampling_rate)
            w_start = w_end - window_samples

            if w_start < 0 or w_end > len(all_ppg_data):
                continue

            ppg_window = np.array(all_ppg_data[w_start:w_end])

            if is_window_bad(ppg_window, sampling_rate):
                print(f"\n[Window #{window_count}] Rejected by SQI — skipping HRV.")
                continue

            current_rmssd, cleaned_peaks = calculate_hrv_rmssd(ppg_window, sampling_rate)

            new_peaks = [p for p in cleaned_peaks if p > last_peak_sample]
            if new_peaks:
                last_peak_sample = max(new_peaks)
            ppg_peaks_data_combined.extend(new_peaks)

            if current_rmssd is not None:
                status = check_hrv_status(current_rmssd, previous_rmssd)
                status_counts[status] += 1
                display_feedback(status, current_rmssd, previous_rmssd, window_count)
                rmssd_log.append((next_calc, current_rmssd))
                previous_rmssd = current_rmssd
            else:
                print(f"\n[Window #{window_count}] Could not compute RMSSD.")

        if data_time >= duration or (stop_event.is_set() and data_queue.empty()):
            print("\n\nPlayback complete!")
            break

    stop_event.set()
    file_thread.join(timeout=5)

    print("\n" + "=" * 60)
    print("SESSION SUMMARY")
    print("=" * 60)
    print(f"Total samples          : {len(all_ppg_data)}")
    print(f"HRV windows analysed   : {window_count}")
    if previous_rmssd:
        print(f"Final RMSSD            : {previous_rmssd:.2f} ms")
    print(f"\nStatus distribution:")
    print(f"  ✓ GREEN  (improving)      : {status_counts[0]} windows")
    print(f"  ~ YELLOW (slight drop)    : {status_counts[1]} windows")
    print(f"  ✗ RED    (significant drop): {status_counts[2]} windows")
    print("=" * 60)

    return {
        "all_ppg":       all_ppg_data,
        "peak_indices":  list(ppg_peaks_data_combined),
        "rmssd_log":     rmssd_log,
        "status_counts": status_counts,
    }


print("Pipeline functions defined.")

## 5. Run Analysis

Execute the full pipeline on `connors_heart.txt`.  The file length is detected automatically so the entire recording is analysed.

In [ ]:
# Determine duration from file length
with open(DATA_FILE) as fh:
    _n_samples = sum(1 for ln in fh if ln.strip())
DURATION = _n_samples // SAMPLING_RATE
print(f"File contains {_n_samples} samples → analysing {DURATION} s of data.\n")

results = collect_and_analyze_from_file(DATA_FILE, duration=DURATION, realtime=False)

## 6. Results Visualization

Three plots are produced:

1. **Full PPG trace** with all detected systolic peaks marked.  
2. **First 10 seconds** — zoomed view to inspect beat-by-beat detection quality.  
3. **RMSSD over time** — HRV trend across the session with colour-coded status bands.

In [ ]:
ppg     = np.array(results["all_ppg"])
peaks   = np.array(results["peak_indices"], dtype=int)
rmssd_t = results["rmssd_log"]   # list of (time_s, rmssd)

t_axis  = np.arange(len(ppg)) / SAMPLING_RATE   # seconds

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle("BME 544 — PPG Signal Quality & HRV Analysis\n"
             f"Dataset: connors_heart.txt  ({len(ppg)} samples @ {SAMPLING_RATE} Hz)",
             fontsize=13, fontweight="bold")

# ── Plot 1: Full signal ─────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(t_axis, ppg, color="#2196F3", linewidth=0.5, label="Raw PPG")
if len(peaks):
    valid_peaks = peaks[peaks < len(ppg)]
    ax1.scatter(t_axis[valid_peaks], ppg[valid_peaks],
                color="#F44336", s=10, zorder=5, label="Detected peaks")
ax1.set_title("Full PPG Recording with Detected Systolic Peaks")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("ADC value")
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: First 10 s zoom ─────────────────────────────────────────────────
ax2 = axes[1]
zoom_samples = 10 * SAMPLING_RATE
zoom_mask    = t_axis <= 10
ax2.plot(t_axis[zoom_mask], ppg[zoom_mask], color="#4CAF50", linewidth=1.2,
         label="Raw PPG (first 10 s)")
if len(peaks):
    zoom_peaks = peaks[(peaks < zoom_samples) & (peaks < len(ppg))]
    if len(zoom_peaks):
        ax2.scatter(t_axis[zoom_peaks], ppg[zoom_peaks],
                    color="#FF5722", s=30, zorder=5, label="Peaks")
ax2.set_title("Zoomed View — First 10 Seconds (Beat Detection Quality)")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("ADC value")
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Plot 3: RMSSD over time ─────────────────────────────────────────────────
ax3 = axes[2]
if rmssd_t:
    times  = [r[0] for r in rmssd_t]
    rmssd_vals = [r[1] for r in rmssd_t]
    ax3.plot(times, rmssd_vals, color="#9C27B0", linewidth=2,
             marker="o", markersize=6, label="RMSSD")
    ax3.axhspan(0,  20,  alpha=0.08, color="red",    label="Low HRV  (< 20 ms)")
    ax3.axhspan(20, 50,  alpha=0.08, color="orange", label="Moderate (20–50 ms)")
    ax3.axhspan(50, max(rmssd_vals + [60]) * 1.15,
                alpha=0.08, color="green",  label="High HRV  (> 50 ms)")
    ax3.set_ylim(0, max(rmssd_vals) * 1.2)
    ax3.set_xlim(0, times[-1] + update_sec)
    for t, r in zip(times, rmssd_vals):
        ax3.annotate(f"{r:.1f}", (t, r), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=8, color="#9C27B0")
else:
    ax3.text(0.5, 0.5, "No valid RMSSD computed", transform=ax3.transAxes,
             ha="center", va="center", fontsize=12)

ax3.set_title("RMSSD Over Time (HRV Trend)")
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("RMSSD (ms)")
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary table ───────────────────────────────────────────────────────────
if rmssd_t:
    rmssd_arr = np.array([r[1] for r in rmssd_t])
    print(f"\n{'─'*40}")
    print(f"  RMSSD statistics across {len(rmssd_arr)} windows")
    print(f"{'─'*40}")
    print(f"  Mean   : {rmssd_arr.mean():.2f} ms")
    print(f"  Std    : {rmssd_arr.std():.2f} ms")
    print(f"  Min    : {rmssd_arr.min():.2f} ms")
    print(f"  Max    : {rmssd_arr.max():.2f} ms")
    print(f"{'─'*40}")